# Elasticsearch Backend — Full-Text + Vector Search with Medha

Medha 0.3.1 ships with `ElasticsearchBackend`: a storage backend that uses
Elasticsearch 8's built-in **kNN dense vector search** to store and retrieve
cache entries. If your team already runs the Elastic stack, this backend lets
you add semantic caching without any new infrastructure.

## When to choose Elasticsearch

| Situation | Recommendation |
|---|---|
| Elastic stack already in production | `elasticsearch` — zero new services |
| Need full-text + vector in one query | `elasticsearch` — combined BM25 + kNN |
| Pure vector-first, no Elastic | `qdrant` — lighter, purpose-built |
| PostgreSQL already in stack | `pgvector` — stay in Postgres |
| Zero external services | `memory` — in-process, no infra |

This notebook covers:
1. **Getting started** — `backend_type="elasticsearch"` in Settings
2. **Store & search** — full waterfall on Elasticsearch
3. **Authentication** — API key and basic auth
4. **Tuning** — `es_num_candidates` impact on recall
5. **TTL & cleanup** — `store(..., ttl=...)` + `expire()`
6. **Backend comparison** — Elasticsearch vs Qdrant vs pgvector
7. **Collection invalidation** — `invalidate_collection()`

**Requirements:**
```bash
pip install "medha-archai[elasticsearch,fastembed]"
```

## Prerequisites — Elasticsearch 8.x via Docker

```bash
# Start a single-node Elasticsearch 8 cluster (security disabled for local dev)
docker run -d --name es8 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  -e "xpack.security.http.ssl.enabled=false" \
  -p 9200:9200 \
  docker.elastic.co/elasticsearch/elasticsearch:8.13.0

# Verify
curl http://localhost:9200/_cluster/health?pretty
```

Set the environment variable used throughout this notebook:
```bash
export MEDHA_TEST_ES_AVAILABLE=1   # set to skip availability check
```

In [ ]:
import asyncio
import os
import time

from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

try:
    from medha import ElasticsearchBackend
    HAS_ES_PKG = True
    print("elasticsearch package is installed")
except ImportError:
    HAS_ES_PKG = False
    print("elasticsearch package not found — install with: pip install medha-archai[elasticsearch]")

# Quick connectivity check
import urllib.request
try:
    urllib.request.urlopen("http://localhost:9200", timeout=2)
    ES_AVAILABLE = True
    print("Elasticsearch reachable at http://localhost:9200")
except Exception:
    ES_AVAILABLE = False
    print("Elasticsearch not reachable — cells requiring ES will be skipped")

CAN_RUN = HAS_ES_PKG and ES_AVAILABLE

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: Settings + Medha with Elasticsearch Backend

In [ ]:
if not CAN_RUN:
    print("Skipping — Elasticsearch not available.")
else:
    settings = Settings(
        backend_type="elasticsearch",
        es_hosts=["http://localhost:9200"],
        es_index_prefix="medha_demo",
    )

    print(f"backend_type    : {settings.backend_type}")
    print(f"es_hosts        : {settings.es_hosts}")
    print(f"es_index_prefix : {settings.es_index_prefix}")
    print(f"es_num_candidates: {settings.es_num_candidates}  (default)")

    medha = Medha("es_demo", embedder=embedder, settings=settings)
    await medha.start()
    print(f"\nBackend: {type(medha._backend).__name__}")
    await medha.close()

## Cell 2: Store 3 Entries + Search + Print CacheHit

In [ ]:
if not CAN_RUN:
    print("Skipping — Elasticsearch not available.")
else:
    settings = Settings(
        backend_type="elasticsearch",
        es_hosts=["http://localhost:9200"],
        es_index_prefix="medha_demo",
    )

    async with Medha("es_demo", embedder=embedder, settings=settings) as medha:
        await medha.store(
            "How many users are registered?",
            "SELECT COUNT(*) FROM users",
        )
        await medha.store(
            "List all active sessions",
            "SELECT * FROM sessions WHERE active = true",
        )
        await medha.store(
            "Average order value this month",
            "SELECT AVG(total) FROM orders WHERE DATE_FORMAT(created_at,'%Y-%m') = DATE_FORMAT(NOW(),'%Y-%m')",
        )

        hit = await medha.search("Total number of registered users")
        print(f"strategy   : {hit.strategy.value}")
        print(f"confidence : {hit.confidence:.4f}")
        print(f"query      : {hit.generated_query}")

## Cell 3: Authentication — API Key, Basic Auth, Timeout

For secured Elasticsearch clusters (Elastic Cloud or on-prem with security enabled),
use one of these patterns. **Only one** auth method should be active at a time.

In [ ]:
# Option A — Elastic Cloud / on-prem with API key
settings_apikey = Settings(
    backend_type="elasticsearch",
    es_hosts=["https://my-cluster.es.io:443"],
    es_api_key="my-base64-encoded-api-key",    # MEDHA_ES_API_KEY
    es_index_prefix="medha_prod",
    es_timeout=60.0,
)
print(f"API key auth  : es_api_key is set = {settings_apikey.es_api_key is not None}")

# Option B — Basic auth (username + password)
settings_basic = Settings(
    backend_type="elasticsearch",
    es_hosts=["https://my-cluster.es.io:443"],
    es_username="elastic",
    es_password="changeme",                    # MEDHA_ES_PASSWORD
    es_timeout=30.0,
)
print(f"Basic auth    : username={settings_basic.es_username!r}")
print(f"Timeout       : {settings_basic.es_timeout}s")

# Environment variable equivalents:
# MEDHA_ES_HOSTS=["http://localhost:9200"]
# MEDHA_ES_API_KEY=...
# MEDHA_ES_USERNAME=elastic
# MEDHA_ES_PASSWORD=changeme
# MEDHA_ES_TIMEOUT=30.0

## Cell 4: Tuning — `es_num_candidates` and Recall

`es_num_candidates` controls how many vectors Elasticsearch's HNSW index inspects
before returning the top-k results. Higher values improve recall at the cost of
latency. The Elasticsearch recommendation is `num_candidates >= 10 × k`.

| `es_num_candidates` | Recall | Latency | Best for |
|---|---|---|---|
| 10–50 | Lower | Fastest | Low-latency, large indices |
| 100 (default) | Good | Balanced | Most production use cases |
| 500–1000 | High | Slower | High-recall requirements |

In [ ]:
if not CAN_RUN:
    print("Skipping — Elasticsearch not available.")
else:
    test_question = "How many users signed up?"

    for num_candidates in [10, 100, 500]:
        settings = Settings(
            backend_type="elasticsearch",
            es_hosts=["http://localhost:9200"],
            es_index_prefix="medha_demo",
            es_num_candidates=num_candidates,
        )
        async with Medha("es_tuning", embedder=embedder, settings=settings) as medha:
            await medha.store("How many users are registered?", "SELECT COUNT(*) FROM users")
            t0 = time.perf_counter()
            hit = await medha.search(test_question)
            elapsed = (time.perf_counter() - t0) * 1000
            print(
                f"  num_candidates={num_candidates:>4d}  "
                f"strategy={hit.strategy.value:<18s}  "
                f"conf={hit.confidence:.4f}  "
                f"{elapsed:.1f}ms"
            )

## Cell 5: TTL + `expire()` — Time-Limited Cache Entries

In [ ]:
if not CAN_RUN:
    print("Skipping — Elasticsearch not available.")
else:
    settings = Settings(
        backend_type="elasticsearch",
        es_hosts=["http://localhost:9200"],
        es_index_prefix="medha_demo",
    )

    async with Medha("es_ttl", embedder=embedder, settings=settings) as medha:
        # Store with a 1-hour TTL
        await medha.store(
            "Today's total revenue",
            "SELECT SUM(amount) FROM orders WHERE DATE(created_at) = CURDATE()",
            ttl=3600,
        )
        # Store with a very short TTL (expires almost immediately)
        await medha.store(
            "Current server time",
            "SELECT NOW()",
            ttl=1,
        )

        print("Stored 2 entries (1h TTL + 1s TTL)")

        await asyncio.sleep(2)  # let the 1s entry expire

        removed = await medha.expire()
        print(f"Removed {removed} expired entries")

## Backend Comparison: Elasticsearch vs Qdrant vs pgvector

| Feature | Elasticsearch | Qdrant | pgvector |
|---|---|---|---|
| **Primary use** | Full-text + vector | Pure vector | SQL + vector |
| **Vector index** | HNSW (built-in) | HNSW + quantization | IVFFlat / HNSW |
| **Quantization** | No | Scalar / Binary | No |
| **Auth** | API key / Basic / OIDC | API key | PostgreSQL roles |
| **Managed cloud** | Elastic Cloud | Qdrant Cloud | RDS, Cloud SQL |
| **Extra infra** | Elasticsearch cluster | Qdrant server | PostgreSQL |
| **Best for** | Elastic stack teams | Vector-first production | PostgreSQL teams |

**Rule of thumb:** choose the backend your team already operates.

## Cell 6: `invalidate_collection()` — Drop All Entries

In [ ]:
if not CAN_RUN:
    print("Skipping — Elasticsearch not available.")
else:
    settings = Settings(
        backend_type="elasticsearch",
        es_hosts=["http://localhost:9200"],
        es_index_prefix="medha_demo",
    )

    async with Medha("es_demo", embedder=embedder, settings=settings) as medha:
        n = await medha.invalidate_collection()
        print(f"invalidate_collection() removed {n} entries")
        count = await medha._backend.count("es_demo")
        print(f"Collection count after invalidation: {count}")